In [ ]:
# ==========================================
# سلول 1: Import و تنظیمات اولیه
# ==========================================
import os
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient
import cohere

# تنظیمات
QDRANT_URL = "http://localhost:6333"
COLLECTION_LAWS = "legal_laws"
COLLECTION_UNITY = "legal_votes_unity"
COLLECTION_DADNAMEH = "legal_votes_dadnameh"

MODEL_EMBED = "baai/bge-m3"

# API Keys
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_EMBEDDINGS_API_KEY")
COHERE_API_KEY = os.environ.get("COHERE_API_KEY")  # باید این را set کنی

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_EMBEDDINGS_API_KEY تنظیم نشده است")
if not COHERE_API_KEY:
    raise ValueError("COHERE_API_KEY تنظیم نشده است")

# کلاینت‌ها
qdrant = QdrantClient(url=QDRANT_URL)
client_embed = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)
co = cohere.Client(api_key=COHERE_API_KEY)

print("✓ همه کلاینت‌ها آماده شدند")

✓ همه کلاینت‌ها آماده شدند


In [3]:
# ==========================================
# سلول 2: تابع گرفتن Embedding
# ==========================================
def embed_query(text: str) -> np.ndarray:
    """تبدیل متن سؤال به بردار embedding"""
    resp = client_embed.embeddings.create(
        model=MODEL_EMBED,
        input=[text]
    )
    emb = np.array(resp.data[0].embedding, dtype="float32")
    return emb

In [4]:
# ==========================================
# سلول 3: تابع Retrieve از سه کالکشن
# ==========================================
def multi_retrieve(query_text: str, top_laws=15, top_unity=10, top_dadnameh=10):
    """
    جستجو در هر سه کالکشن و ترکیب نتایج
    
    Args:
        query_text: متن سؤال
        top_laws: تعداد نتایج از کالکشن قوانین
        top_unity: تعداد نتایج از وحدت رویه
        top_dadnameh: تعداد نتایج از دادنامه‌ها
    
    Returns:
        لیست دیکشنری‌های نتایج
    """
    # Embedding سؤال
    q_vec = embed_query(query_text).tolist()
    
    all_results = []
    
    # 1. جستجو در قوانین
    try:
        hits = qdrant.query_points(
            collection_name=COLLECTION_LAWS,
            query=q_vec,
            limit=top_laws,
            with_payload=True
        ).points
        
        for h in hits:
            all_results.append({
                "source_type": "قانون",
                "retrieval_score": h.score,
                "text": h.payload.get("page_content", "")[:3500],  # محدود به 3500 کاراکتر
                "law_name": h.payload.get("law_name", "نامشخص"),
                "article_number": h.payload.get("article_number", ""),
                "metadata": {
                    "law": h.payload.get("law_name", ""),
                    "article": h.payload.get("article_number", "")
                }
            })
    except Exception as e:
        print(f"خطا در جستجوی قوانین: {e}")
    
    # 2. جستجو در وحدت رویه
    try:
        hits = qdrant.query_points(
            collection_name=COLLECTION_UNITY,
            query=q_vec,
            limit=top_unity,
            with_payload=True
        ).points
        
        for h in hits:
            all_results.append({
                "source_type": "وحدت رویه",
                "retrieval_score": h.score,
                "text": h.payload.get("page_content", "")[:3500],
                "law_name": "وحدت رویه",
                "article_number": h.payload.get("ruling_number", ""),
                "metadata": {
                    "ruling": h.payload.get("ruling_number", ""),
                    "date": h.payload.get("date", "")
                }
            })
    except Exception as e:
        print(f"خطا در جستجوی وحدت رویه: {e}")
    
    # 3. جستجو در دادنامه‌ها
    try:
        hits = qdrant.query_points(
            collection_name=COLLECTION_DADNAMEH,
            query=q_vec,
            limit=top_dadnameh,
            with_payload=True
        ).points
        
        for h in hits:
            all_results.append({
                "source_type": "دادنامه",
                "retrieval_score": h.score,
                "text": h.payload.get("page_content", "")[:3500],
                "law_name": "دادنامه",
                "article_number": h.payload.get("ruling_number", ""),
                "metadata": {
                    "ruling": h.payload.get("ruling_number", ""),
                    "date": h.payload.get("date", "")
                }
            })
    except Exception as e:
        print(f"خطا در جستجوی دادنامه‌ها: {e}")
    
    return all_results


In [5]:
# ==========================================
# سلول 4: تابع Reranking با Cohere
# ==========================================
def rerank_with_cohere(query: str, results: list, top_k=10, model="rerank-multilingual-v3.5"):
    """
    Rerank نتایج با استفاده از Cohere API
    
    Args:
        query: متن سؤال
        results: لیست نتایج از multi_retrieve
        top_k: تعداد نتایج نهایی
        model: مدل Cohere (rerank-multilingual-v3 یا v3.5)
    
    Returns:
        لیست rerank شده نتایج
    """
    if len(results) == 0:
        return []
    
    # اگر نتایج کمتر از top_k است، همه را برگردان
    if len(results) <= top_k:
        return results
    
    # آماده‌سازی documents برای Cohere
    documents = [r["text"] for r in results]
    
    try:
        # فراخوانی Cohere Rerank API
        response = co.rerank(
            query=query,
            documents=documents,
            model=model,
            top_n=top_k,
            return_documents=False  # چون خودمان متن را داریم
        )
        
        # ترکیب نتایج rerank شده با metadata اصلی
        reranked_results = []
        for result in response.results:
            idx = result.index
            original = results[idx].copy()
            original["rerank_score"] = result.relevance_score
            original["rerank_position"] = result.index
            reranked_results.append(original)
        
        return reranked_results
        
    except Exception as e:
        print(f"خطا در Cohere Rerank: {e}")
        # در صورت خطا، نتایج اولیه را بر اساس retrieval score مرتب برگردان
        return sorted(results, key=lambda x: x["retrieval_score"], reverse=True)[:top_k]


In [6]:
# ==========================================
# سلول 5: تابع نمایش نتایج
# ==========================================
def display_results(query: str, results: list, show_text_length=200):
    """نمایش زیبای نتایج"""
    print("=" * 80)
    print("سؤال:")
    print(query.strip())
    print("=" * 80)
    print(f"\nتعداد نتایج: {len(results)}\n")
    
    for i, r in enumerate(results, 1):
        print(f"\n{'─' * 80}")
        print(f"رتبه {i} | نوع: {r['source_type']}")
        
        if "rerank_score" in r:
            print(f"امتیاز Rerank: {r['rerank_score']:.4f} | امتیاز Retrieval: {r['retrieval_score']:.4f}")
        else:
            print(f"امتیاز: {r['retrieval_score']:.4f}")
        
        # نمایش metadata
        if r['law_name'] and r['law_name'] != "نامشخص":
            meta_str = f"منبع: {r['law_name']}"
            if r.get('article_number'):
                meta_str += f" | شماره: {r['article_number']}"
            print(meta_str)
        
        # نمایش بخشی از متن
        text_preview = r['text'][:show_text_length].replace("\n", " ")
        print(f"متن: {text_preview}...")
        
    print("\n" + "=" * 80)


In [7]:
# ==========================================
# سلول 6: تابع تست کامل (با و بدون Rerank)
# ==========================================
def test_query_complete(query_text: str, use_rerank=True, top_k=5):
    """
    تست کامل یک سؤال با Retrieval و Reranking
    
    Args:
        query_text: متن سؤال
        use_rerank: استفاده از Cohere Rerank یا نه
        top_k: تعداد نتایج نهایی
    """
    print("\n🔍 شروع جستجو...")
    
    # مرحله 1: Retrieve از سه کالکشن
    results = multi_retrieve(query_text, top_laws=15, top_unity=10, top_dadnameh=10)
    print(f"✓ تعداد نتایج اولیه: {len(results)}")
    
    # مرحله 2: Rerank (اختیاری)
    if use_rerank:
        print("🔄 Reranking با Cohere...")
        final_results = rerank_with_cohere(query_text, results, top_k=top_k)
        print(f"✓ Rerank انجام شد")
    else:
        # بدون rerank، فقط بر اساس retrieval score مرتب کن
        final_results = sorted(results, key=lambda x: x["retrieval_score"], reverse=True)[:top_k]
    
    # نمایش نتایج
    display_results(query_text, final_results)
    
    return final_results


In [ ]:
# ==========================================
# سلول 7: تابع تست کامل با Hybrid + Rerank
# ==========================================
def test_query_hybrid(query_text: str, use_hybrid=True, use_rerank=True, top_k=5, alpha=0.7):
    """
    تست کامل با Hybrid Retrieval + Reranking
    
    Args:
        query_text: متن سؤال
        use_hybrid: اگر False باشد، فقط dense استفاده می‌شود
        use_rerank: استفاده از Cohere Rerank
        top_k: تعداد نتایج نهایی
        alpha: وزن dense در hybrid (0.7 = 70% dense, 30% sparse)
    """
    print("\n🔍 شروع جستجو...")
    
    # مرحله 1: Retrieve
    if use_hybrid:
        print("🔀 Hybrid Retrieval (Dense + Sparse)...")
        results = multi_retrieve_hybrid(query_text, alpha=alpha)
    else:
        print("📊 Dense-only Retrieval...")
        results = multi_retrieve(query_text)  # تابع قبلی
    
    print(f"✓ تعداد نتایج اولیه: {len(results)}")
    
    # مرحله 2: Rerank
    if use_rerank:
        print("🔄 Reranking با Cohere...")
        final_results = rerank_with_cohere(query_text, results, top_k=top_k)
        print(f"✓ Rerank انجام شد")
    else:
        final_results = results[:top_k]
    
    # نمایش
    display_results(query_text, final_results)
    
    return final_results



🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 
تست با Cohere Rerank
🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 🟢 


سؤال 1 از 3

🔍 شروع جستجو...
✓ تعداد نتایج اولیه: 35
🔄 Reranking با Cohere...
✓ Rerank انجام شد
سؤال:
شرکتی با اختراع یک دستگاه و فروش عمده آن به کارخانجات خودروسازی موجب بیکار شدن تعداد زیادی از کارگران کارخانجات مزبور می.شود کدام مورد در خصوص اقدام مذکور، صحیح است؟ 
1) این اقدام مغایر اصل 40  قانون اساسی دایر بر ممنوعیت اضرار به غیر به واسطه اعمال حق خویش است. 
2) با توجه به شناسایی آزادی کسب و کار در صدر اصل 28  قانون اساسی ایراد قانونی بر آن مترتب  نیست. 
3) عمل مذکور از مصادیق ذیل اصل 46 قانون اساسی مبنی بر ممنوعیت سلب کسب و کار از دیگری به بهانه مالکیت   نسبت  به کسب و  کار خود است . 
4) اقدام شرکت بر اساس اصل 28  قانون اساسی که آزادی اشتغال را محدود به مصالح عمومی می داند مغایر قانون اساسی است.

تعداد نتایج: 5


────────────────────────────────────────────────────────────────────────────────
رتبه 1 | نوع: قانون
امتیاز Rerank: 0.4660 | 

In [ ]:
# ==========================================
# سلول 8: تست و مقایسه
# ==========================================

q_test = """
کدام مورد در خصوص تفویض اختیار قانونگذاری از سوی مجلس شورای اسلامی، صحیح است؟
"""

print("\n" + "="*80)
print("مقایسه روش‌های مختلف")
print("="*80)

# 1. Dense-only (فعلی شما)
print("\n\n🔵 روش 1: Dense-only + Cohere")
r1 = test_query_complete(q_test, use_rerank=True, top_k=5)

# 2. Hybrid (Dense + Sparse)
print("\n\n🟣 روش 2: Hybrid (70% Dense + 30% Sparse) + Cohere")
r2 = test_query_hybrid(q_test, use_hybrid=True, use_rerank=True, top_k=5, alpha=0.7)

# 3. تست با alpha متفاوت
print("\n\n🟠 روش 3: Hybrid (50% Dense + 50% Sparse) + Cohere")
r3 = test_query_hybrid(q_test, use_hybrid=True, use_rerank=True, top_k=5, alpha=0.5)


برای دیدن میزان استفاده، به https://dashboard.cohere.com بروید
